# Решения: Интеграция: библиотека алгоритмов и сложность

**Для преподавателя.** Полный эталон к `lesson.ipynb` и `homework.ipynb`; ученикам до сдачи не показывать.

In [ ]:
from pathlib import Path
import math
import statistics
import time
import pandas as pd


def find_csv(name):
    for path in (Path(name), Path("../../data") / name, Path("../data") / name):
        if path.exists():
            return path.resolve()
    raise FileNotFoundError(f"{name} не найден рядом с ноутбуком или в data/")


unsorted_df = pd.read_csv(find_csv("bank_transactions_unsorted.csv"))
by_id_df = pd.read_csv(find_csv("bank_transactions_sorted_by_txn_id.csv"))
by_amount_df = pd.read_csv(find_csv("bank_transactions_sorted_by_amount.csv"))
tiny_df = pd.read_csv(find_csv("bank_transactions_tiny.csv"))
COLS = ["txn_id", "amount", "day", "risk_score"]
unsorted_txns = list(unsorted_df[COLS].itertuples(index=False, name=None))
id_txns = list(by_id_df[COLS].itertuples(index=False, name=None))
amount_txns = list(by_amount_df[COLS].itertuples(index=False, name=None))
tiny_txns = list(tiny_df[COLS].itertuples(index=False, name=None))
id_list = [row[0] for row in id_txns]
amount_list = [row[1] for row in amount_txns]
assert id_list == sorted(id_list)
assert amount_list == sorted(amount_list)
print(f"Загружено {len(unsorted_txns)} транзакций; поля кортежа: {COLS}")


## Урок. 1–3. Полный API

In [ ]:
def linear_search(values, target):
    for index, value in enumerate(values):
        if value == target:
            return index
    return -1


def binary_search(values, target):
    left, right = 0, len(values) - 1
    while left <= right:
        mid = (left + right) // 2
        if values[mid] == target:
            return mid
        if values[mid] < target:
            left = mid + 1
        else:
            right = mid - 1
    return -1


def lower_bound(values, target):
    left, right = 0, len(values)
    while left < right:
        mid = (left + right) // 2
        if values[mid] < target:
            left = mid + 1
        else:
            right = mid
    return left


def upper_bound(values, target):
    left, right = 0, len(values)
    while left < right:
        mid = (left + right) // 2
        if values[mid] <= target:
            left = mid + 1
        else:
            right = mid
    return left


def selection_sort(values):
    result = list(values)
    for i in range(len(result)):
        smallest = i
        for j in range(i + 1, len(result)):
            if result[j] < result[smallest]:
                smallest = j
        result[i], result[smallest] = result[smallest], result[i]
    return result


def merge_sorted(left, right):
    i = j = 0
    result = []
    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            result.append(left[i]); i += 1
        else:
            result.append(right[j]); j += 1
    return result + list(left[i:]) + list(right[j:])


def merge_sort(values):
    if len(values) <= 1:
        return list(values)
    mid = len(values) // 2
    return merge_sorted(merge_sort(values[:mid]), merge_sort(values[mid:]))


def median_runtime(function, values, repeats=3):
    samples = []
    for _ in range(repeats):
        start = time.perf_counter()
        function(list(values))
        samples.append(time.perf_counter() - start)
    return statistics.median(samples)

API = ["linear_search", "binary_search", "lower_bound", "upper_bound", "selection_sort", "merge_sort"]
assert len(API) == 6


## Урок. 4. Quality gate

In [ ]:
cases = [[], [1], [2, 1], [3, 1, 3], list(range(25, -1, -1))]
probe = [3, 1, 3]; selection_sort(probe); merge_sort(probe)
quality = {"linear": linear_search([1, 2], 3) == -1, "binary": binary_search([1, 2], 2) == 1, "bounds": lower_bound([1, 2, 2], 2) == 1 and upper_bound([1, 2, 2], 2) == 3, "selection": all(selection_sort(c) == sorted(c) for c in cases), "merge": all(merge_sort(c) == sorted(c) for c in cases), "input_preserved": probe == [3, 1, 3]}
assert set(quality.values()) == {True}


## Урок. 5. Benchmark

In [ ]:
sizes = [80, 160, 320, 640]; benchmark_rows = []
for n in sizes:
    values = [r[1] for r in unsorted_txns[:n]]
    ts = median_runtime(selection_sort, values); tm = median_runtime(merge_sort, values)
    benchmark_rows.append({"n": n, "selection_s": ts, "merge_s": tm, "ratio": ts / tm if tm else 0.0})
benchmark = pd.DataFrame(benchmark_rows)
assert len(benchmark) == 4


## Урок. 6. Range query

In [ ]:
lo, hi = lower_bound(amount_list, 10000), upper_bound(amount_list, 20000)
range_rows = amount_txns[lo:hi]
assert range_rows == [r for r in amount_txns if 10000 <= r[1] <= 20000]


## Урок. 7–9. Acceptance и REPORT

In [ ]:
REPORT = "Библиотека реализует линейный и бинарный поиск, границы диапазона, selection sort и mergesort. Quality gate проверяет пустые списки, дубликаты, отсутствие цели и сохранение входа. Benchmark на четырёх размерах даёт evidence согласованного роста: selection соответствует O(n²), merge — O(n log n), а встроенный sorted остаётся production baseline. Диапазонный запрос проверен прямой фильтрацией. Ограничение: учебные размеры, конкретная машина и шум времени не доказывают асимптотику и не объясняют банковский риск."
acceptance = pd.Series({"api_complete": len(API) == 6, "quality_gate": all(quality.values()), "benchmark_4_sizes": len(benchmark) == 4, "range_query": range_rows == [r for r in amount_txns if 10000 <= r[1] <= 20000], "report_ready": len(REPORT) >= 350})
READY = bool(acceptance.all())
artifact_manifest = {"library": "bank_logs.py", "benchmark": "benchmark.csv", "report": "REPORT.md", "ready": READY}
assert READY is True


## ДЗ. Part A

In [ ]:
new_cases = [[0, -1, 0], [5] * 20, list(range(50))]
new_gate = [fn(c) == sorted(c) for c in new_cases for fn in (selection_sort, merge_sort)]
risk_rows = []
for n in (100, 300, 600):
    values = [r[3] for r in unsorted_txns[:n]]; ts = median_runtime(selection_sort, values); tm = median_runtime(merge_sort, values)
    risk_rows.append([n, ts, tm, ts / tm if tm else 0.0])
builtin_rows = [(n, median_runtime(sorted, [r[3] for r in unsorted_txns[:n]])) for n in (100, 300, 600)]
assert all(new_gate) and len(risk_rows) == len(builtin_rows) == 3


## ДЗ. Challenge

In [ ]:
def audit_algorithms(values, targets):
    ordered = merge_sort(values)
    positions = [binary_search(ordered, target) for target in targets]
    return {"sorted": ordered, "positions": positions, "quality": ordered == sorted(values) and values == list(values)}

result = audit_algorithms([5, 1, 3, 3], [1, 3, 9])
REFLECTION = "Блок связал поиск с предпосылкой порядка: линейный поиск универсален, бинарный использует сортировку. Selection и mergesort показали разный рост, а два указателя превратили порядок в правило движения без вложенного перебора. Решение принимается по evidence тестов и benchmark, не по одному замеру. Ограничение: учебные данные и время машины не заменяют production profiling и бизнес-проверку."
assert result["positions"] == [0, 1, -1] and len(REFLECTION) >= 300
